### Register & Deploy an ML model in Azure Machine Learning / Foundry



### Goal
The goal of this notebook is to demonstrate how to deploy a custom model with a custom VLLM container into  Azure Machine Learning managed online endpoints for inferencing. The steps include creating a custom environment, registering the model, and deploying it to an online endpoint.

### Prerequisites

* To use Azure Machine Learning, you must have an Azure subscription. 
* You must have an Azure resource group, and you (or the service principal you use) must have Contributor access to it.
* You must have an Azure Machine Learning workspace
* You must have the Azure Machine Learning SDK for Python installed. You can install it using pip:
```bash
pip install azure-ai-ml
# Set your workspace as default
az configure --defaults group=$GROUP workspace=$WORKSPACE location=$LOCATION
```


In [ ]:
# import required libraries
from azure.ai.ml import MLClient
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    Model,
    Environment
)
from azure.identity import DefaultAzureCredential

## 1.2. Configure workspace details and get a handle to the workspace

To connect to a workspace, we need identifier parameters - a subscription, resource group and workspace name. We will use these details in the `MLClient` from `azure.ai.ml` to get a handle to the required Azure Machine Learning workspace. We use the default [default azure authentication](https://docs.microsoft.com/en-us/python/api/azure-identity/azure.identity.defaultazurecredential?view=azure-python). 

In [ ]:
# First lets get the default workspace, RG and subscription from Azure CLI config
# This assumes you have already configured your Azure CLI with the correct defaults
# and have run `az configure` to set the defaults for your Azure CLI commands.
import os
import configparser

azure_config_path = os.path.expanduser("~/.azure/config")

# Load the config file
config = configparser.ConfigParser()
config.read(azure_config_path)

# Extract defaults
defaults = {}
if 'defaults' in config:
    for key in config['defaults']:
        defaults[key] = config['defaults'][key]



In [ ]:
# enter details of your AML workspace
subscription_id = defaults['subscription']
resource_group = defaults['group']
workspace = defaults['workspace']

In [ ]:
# get a handle to the workspace
ml_client = MLClient(
    DefaultAzureCredential(), subscription_id, resource_group, workspace
)

## Create a custom environment

### Option 1: Use Container built locally and pushed to ACR in the workspace
This is often a faster way to iterate since building containers in Azure ML can take more time esp for minor updates.

The container can be derived off standard containers (like vllm pre built images) with no Azure ML specific dependencies. The only thing Azure ML needs is an utility called "runit" which is used to start the container and run the model server and can restart if inference server crashes without restarting whole inference instances. 

Refer to ./docker/Dockerfile for how to add runit and sample of inference server (like VLLM) launcher in "docker/runit_folder/api_server/run"

Assumption: You have already built the container locally and pushed it to your ACR in the workspace. 

```python




In [ ]:
# Get ACR Name
workspace_details = ml_client.workspaces.get(ml_client.workspace_name)

# Extract ACR resource ID
acr_resource_id = workspace_details.container_registry
acr_name = acr_resource_id.split("/")[-1]


In [ ]:
from azure.ai.ml.entities import Environment, ProbeSettings
custom_env = Environment(name="my-vllm-v5", 
                         image=f"{acr_name}.azurecr.io/custom/vllm0.2.7:v5", # replace with your ACR image 
                         description="vllm custom environment",
                         inference_config={
                             "liveness_route": {"path": "/health", "port": 8000},
                             "readiness_route": {"path": "/health", "port": 8000},
                             "scoring_route": {"path": "/", "port": 8000}
                         },
)
custom_env

### Option 2: Create a container image using Azure ML (This is slower)


In [ ]:
#from azure.ai.ml.entities import Environment, BuildContext
# Create a custom environment for vLLM using a Dockerfile
#custom_env = Environment(
#    name="hf-vllm-gpu",
#    description="Environment with vLLM and dependencies using Dockerfile",
#    build=BuildContext(path="./docker", dockerfile_path="Dockerfile"),
#)

#ml_client.environments.create_or_update(custom_env)


## Register Model

In [ ]:
import os
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes


os.environ["AZUREML_ARTIFACTS_DEFAULT_TIMEOUT"] = "3000"

# Location of your model on local machine. Replace with your model path.
artifact_path = "./SmolLM2-135M-hf"
model_name = "SmolLM2-135M-hf"

# Register the model using Azure ML SDK
registered_model = ml_client.models.create_or_update(
    Model(
        path=artifact_path,
        name=model_name,
        description="SmolLM2-135M custom registered model",
        type=AssetTypes.CUSTOM_MODEL  # Set type if needed, e.g., 'custom_model'
    )
)
print(f"Registered model: {registered_model.name}:{registered_model.version}")

In [ ]:
# Get the registered model by name and label. 
# DO THIS IF YOU DONT WANT TO REGISTER AGAIN. Comment this otherwise
model_name="SmolLM2-135M-hf"
registered_model = ml_client.models.get(name=model_name, label="latest")
#if not models:
#model_list = [f"{model.name}:{model.version}" for model in models]
registered_model

## Custom Deployment with Azure ML SDK

### Create Endpoint

In [ ]:
from azure.ai.ml.entities import ManagedOnlineEndpoint
import uuid

# Unique endpoint name
endpoint_name = f"hf-llm-endpoint-{uuid.uuid4().hex[:6]}"


endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Hugging Face LLM endpoint",
    auth_mode="key"
)
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"Created endpoint: {endpoint_name}")

### Create Deployment on the Endpoint

In [ ]:
from azure.ai.ml.entities import ManagedOnlineDeployment, ProbeSettings

# uncomment next line if reusing endpoint
endpoint_name = "hf-llm-endpoint-5312ec"

deployment = ManagedOnlineDeployment(
    name="default",
    endpoint_name=endpoint_name,
    model=registered_model.id,
    environment=custom_env,
    #model_mount_path="/models",
    environment_variables={
        "MODEL_PATH": "SmolLM2-135M-hf",
        "VLLM_ARGS": "--dtype float16 --enforce-eager"
    },
    instance_type="Standard_NC6s_v3", # Replace with your desired instance type
    instance_count=1
)

ml_client.online_deployments.begin_create_or_update(deployment).result()
print("Deployment created.")

In [ ]:
# Assign 100% of traffic to the deployment

endpoint = ml_client.online_endpoints.get(name=endpoint_name)
endpoint.traffic = {"default": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Traffic updated.")

In [ ]:
# Get the details for online endpoint
endpoint_name = "hf-llm-endpoint-5312ec"
endpoint = ml_client.online_endpoints.get(name=endpoint_name)
print(endpoint.traffic)
print(endpoint.scoring_uri)
endpoint

##  Get the logs for the new deployment
Get the logs for the green deployment and verify as needed

In [ ]:
ml_client.online_deployments.get_logs(
    name="default", endpoint_name=endpoint_name, lines=50
)

## Run a inference

In [ ]:
import requests
url = f"{endpoint.scoring_uri}/v1/models" # May have to drop the "/score" at end
api_key = ml_client.online_endpoints.get_keys(name=endpoint_name).primary_key

if not api_key:
    raise Exception("A key should be provided to invoke the endpoint")

headers = {
    "Authorization": f"Bearer {api_key}"
}

response = requests.get(url, headers=headers)
if response.status_code == 200:
    models = response.json()
    print(models)
else:
    print(f"Error: {response.status_code}, {response.text}")


In [ ]:
import requests
import json

data = {
    "model": "/var/azureml-app/azureml-models/SmolLM2-135M-hf/2/SmolLM2-135M-hf", # Replace with your model path
    "prompt": "Seattle is a",
    "max_tokens": 200,
    "temperature": 0.7
}

# Use the endpoint's scoring URI
url = f"{endpoint.scoring_uri}/v1/completions" # May have to drop the "/score" at end
api_key = ml_client.online_endpoints.get_keys(name=endpoint_name).primary_key

if not api_key:
    raise Exception("A key should be provided to invoke the endpoint")

headers = {'Content-Type':'application/json', 'Accept': 'application/json', 'Authorization':('Bearer '+ api_key)}

print(f"{url=}, {api_key=}, {data=}, {headers=}")

response = requests.post(url, headers=headers, json=data)
print(response.json())


# Delete the endpoint


In [ ]:
#ml_client.online_endpoints.begin_delete(name=endpoint_name)